# CS2 Integrity Watch: load data on LinuxONE

Loads the prepared CS2 tables: economy, per-player stats, rounds, kills and utility bought for
**1,988 professional maps** (860 matches, 70 events, Mar 23 - Apr 20 2026).

**Setup:** upload the 8 `.csv.gz` files into `cs2_betting_data/tables/` (the folder next to `IBMZProject`).
They were built off-VM with `build_tables_local.py`, because `pyarrow` (needed to read the source Parquet)
cannot be built on s390x. Pandas reads gzipped CSV natively, so nothing needs installing here.

| Source | License |
|---|---|
| [sneakyzero/cs2-pro-round-economy](https://huggingface.co/datasets/sneakyzero/cs2-pro-round-economy) | CC BY 4.0 |
| [blanchon/opencs2_dataset_demo](https://huggingface.co/datasets/blanchon/opencs2_dataset_demo) (`*.analysis.json`) | CC BY 4.0 |

Demos originate from HLTV's public demo downloads. Credit both datasets and HLTV in the submission.

In [ ]:
# 1. Environment
import os, sys, platform
import pandas as pd
print("Python :", sys.version.split()[0], "| machine:", platform.machine(), "| byte order:", sys.byteorder)
print("pandas :", pd.__version__)

In [ ]:
# 2. Locate the tables (cs2_betting_data/tables next to the repo, never inside it)
TABLES = next((p for p in ("../cs2_betting_data/tables", "cs2_betting_data/tables") if os.path.isdir(p)), None)
assert TABLES, "Create cs2_betting_data/tables next to IBMZProject and upload the .csv.gz files there"
TABLES = os.path.abspath(TABLES)
print(TABLES)
print(sorted(f for f in os.listdir(TABLES) if f.endswith(".csv.gz")))

In [ ]:
# 3. Load. Steam IDs and match IDs stay strings (Steam IDs exceed float precision)
ID_COLS = {c: "string" for c in ("steamid", "attacker_steamid", "victim_steamid", "assister_steamid",
                                  "match_id", "map_uid", "roster_uid", "ct_roster_uid", "t_roster_uid")}

def load(name):
    df = pd.read_csv(os.path.join(TABLES, f"{name}.csv.gz"), dtype=ID_COLS, low_memory=False)
    print(f"{name:22s} {df.shape}")
    return df

round_table  = load("round_table")           # one row per round: both teams' economy + winner
player_stats = load("player_map_stats")      # one row per player per side per map: ADR, KAST, rating...
round_detail = load("round_detail")          # round end reason, bomb plant, ticks
kills        = load("kills")                 # every kill: attacker, victim, weapon, headshot...
util_bought  = load("util_bought")           # grenades held per player per round
player_econ  = load("player_round_economy")  # each player's cash, spend, equipment per round
pistol_opps  = load("pistol_loss_opps")      # rounds right after a lost pistol round
maps_meta    = load("maps_meta")             # one row per map: teams, event, score, QC

round_table = round_table[round_table["eligible"]]
print("eligible rounds:", len(round_table))

In [ ]:
# 4. Sanity check: the richer team should win more often, and more so as the gap grows
rt = round_table.copy()
gap = rt["equip_diff_ct_minus_t"]
rt = rt[gap != 0].assign(
    richer_won=lambda d: (d["equip_diff_ct_minus_t"] > 0) == (d["winner_side"] == "ct"),
    gap=lambda d: pd.cut(d["equip_diff_ct_minus_t"].abs(), [0, 2000, 5000, 10000, 20000, 100000],
                         labels=["<2k", "2-5k", "5-10k", "10-20k", "20k+"]))
rt.groupby("gap", observed=True)["richer_won"].agg(["mean", "count"]).rename(
    columns={"mean": "richer_team_win_rate", "count": "rounds"})

In [ ]:
# 5. First integrity signal: which teams lose rounds they were heavily favored to win?
#    "Favored" = at least $15k more equipment than the opponent at freeze end.
BIG_EDGE = 15000
edge = round_table["equip_diff_ct_minus_t"]
fav = round_table[edge.abs() >= BIG_EDGE].copy()
fav["favored_team"] = fav["ct_team_name_resolved"].where(edge[fav.index] > 0, fav["t_team_name_resolved"])
fav["favored_lost"] = fav["favored_team"] != fav["winner_team"]

baseline = fav["favored_lost"].mean()
by_team = (fav.groupby("favored_team")["favored_lost"].agg(["sum", "count"])
              .rename(columns={"sum": "favored_rounds_lost", "count": "favored_rounds"}))
by_team["loss_rate"] = by_team["favored_rounds_lost"] / by_team["favored_rounds"]
print(f"Across all teams, the favored side loses {baseline:.1%} of these rounds.")
# Teams with enough favored rounds to judge, highest loss rate first (a starting point, not an accusation)
by_team[by_team["favored_rounds"] >= 20].sort_values("loss_rate", ascending=False).head(15)

## Next steps

- **Expected-outcome model:** round win probability from both teams' equipment, side, score and map
  (XGBoost, then TensorFlow on the on-chip accelerator). Flag rounds and maps that miss the prediction by far.
- **Player model:** expected ADR/KAST/rating for each player against each opponent; flag large shortfalls.
- **Season test:** does a team or player lose favored rounds more often than chance allows? (binomial test,
  corrected for testing many teams). Cell 5 is the crude version of this.
- **Utility *used*** (thrown, enemies flashed, team flashes) is not in these tables: take it from bo3.gg's
  per-map Grenades tab (credit with a link), or run `extract_utility_local.py` on a laptop for chosen matches.